In [1]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, adj_price, divend에 대해 반환
def extract_stock_data(file_path, ticker):
    extract_datas = []

    with open(os.path.join(file_path, ticker + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    adj_price_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    adj_price_history = list(map(float, extract_datas[3]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[4]))
    divend_history = list(map(float, extract_datas[5]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    adj_price_df = pd.DataFrame({'date': adj_price_dates, 'adj_price': adj_price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    adj_price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, adj_price_df, divend_df)

def make_stock_data(file_path, tickers):
    stock_data = []

    for ticker in tickers:
        price_df, adj_price_df, divend_df = extract_stock_data(file_path, ticker)
        stock_df = pd.concat([price_df, adj_price_df, divend_df], axis=1, sort=True)
        stock_df.columns = pd.MultiIndex.from_product([[ticker], stock_df.columns])
        stock_data.append(stock_df)

    stock_data = pd.concat(stock_data, axis=1, sort=True)

    return stock_data


In [2]:
from portfolio import Portfolio

# stock 불러와서 데이터프레임화 하기
file_path = './etf'
tickers = ['QQQ', 'DGRW', 'SCHD', 'SPY', 'SCHG', 'SPYG']
stock_data = make_stock_data(file_path, tickers)
pd.options.display.float_format = '{:.2f}'.format

# test_stock = Portfolio(make_stock_data(file_path, ['QQQ', 'SCHG']), 10000)
test_stock = Portfolio(stock_data, 10000)
portfolio = Portfolio(stock_data, 10000, {"QQQ":56, "SCHD":24, "DGRW":20})

In [3]:
def calc_target_ratio(base_ratio:tuple, etc_ratio:tuple) -> tuple:
    base_sum = sum(base_ratio)
    qqq = base_ratio[0] / base_sum * 100
    etc_sum = sum(etc_ratio)
    etc1 = (etc_ratio[0] / etc_sum) * (base_ratio[1] / base_sum) * 100
    etc2 = (etc_ratio[1] / etc_sum) * (base_ratio[1] / base_sum) * 100

    return (round(qqq, 1), round(etc1, 1), round(etc2, 1))

In [4]:
import numpy as np
p = Portfolio(stock_data, 10000, {'QQQ':50, 'SCHD':30, 'DGRW':20})
history = p.backtest(duration=5, rebalancing_cycle=5, cash_flow=100, cash_flow_growth=10)
np.set_printoptions(threshold=np.inf)
history

[[10000 0 0 ... np.float64(80.321285) 2000.0 20.0]
 [np.float64(9974.0497906) np.float64(0.0) np.float64(0.0) ...
  np.float64(80.321285) np.float64(1993.5742937)
  np.float64(19.98761120662176)]
 [np.float64(9976.65228646) np.float64(0.0) np.float64(0.0) ...
  np.float64(80.321285) np.float64(1993.5742937)
  np.float64(19.98239726572025)]
 ...
 [np.float64(20224.692747965742) np.float64(1159.7088318257456)
  np.float64(5.734123362355616) ... np.float64(80.321285)
  np.float64(3314.0562191) np.float64(16.38618821259145)]
 [np.float64(20360.002949465746) np.float64(1159.7088318257456)
  np.float64(5.696015048250161) ... np.float64(80.321285)
  np.float64(3342.9718817) np.float64(16.41930941757413)]
 [np.float64(20305.060930295745) np.float64(1162.9216832257457)
  np.float64(5.727250399385074) ... np.float64(80.321285)
  np.float64(3323.6947733) np.float64(16.368799801732926)]]
(1260, 15)


total  cash           QQQ                        SCHD         \
            value value weight  price number   value weight price number   
date                                                                       
2013-05-22  10000     0      0  73.62  67.92 5000.00  50.00 11.28 265.96   
2013-05-23    NaN   NaN    NaN  73.45    NaN     NaN    NaN 11.25    NaN   
2013-05-24    NaN   NaN    NaN  73.41    NaN     NaN    NaN 11.27    NaN   
2013-05-28    NaN   NaN    NaN  73.89    NaN     NaN    NaN 11.34    NaN   
2013-05-29    NaN   NaN    NaN  73.54    NaN     NaN    NaN 11.20    NaN   
...           ...   ...    ...    ...    ...     ...    ...   ...    ...   
2018-05-16    NaN   NaN    NaN 168.98    NaN     NaN    NaN 16.48    NaN   
2018-05-17    NaN   NaN    NaN 168.33    NaN     NaN    NaN 16.51    NaN   
2018-05-18    NaN   NaN    NaN 167.46    NaN     NaN    NaN 16.46    NaN   
2018-05-21    NaN   NaN    NaN 168.40    NaN     NaN    NaN 16.62    NaN   
2018-05-22    NaN   NaN    NaN 168.18    NaN     NaN    NaN 16.53    NaN   

                           DGRW                        
             value weight price number   value weight  
date                                                   
2013-05-22 3000.00  30.00 24.90  80.32 2000.00  20.00  
2013-05-23     NaN    NaN 24.82    NaN     NaN    NaN  
2013-05-24     NaN    NaN 24.82    NaN     NaN    NaN  
2013-05-28     NaN    NaN 25.03    NaN     NaN    NaN  
2013-05-29     NaN    NaN 24.84    NaN     NaN    NaN  
...            ...    ...   ...    ...     ...    ...  
2018-05-16     NaN    NaN 41.27    NaN     NaN    NaN  
2018-05-17     NaN    NaN 41.31    NaN     NaN    NaN  
2018-05-18     NaN    NaN 41.26    NaN     NaN    NaN  
2018-05-21     NaN    NaN 41.62    NaN     NaN    NaN  
2018-05-22     NaN    NaN 41.38    NaN     NaN    NaN  

[1260 rows x 15 columns]

In [ ]:
h.to_csv('./abc.csv')

In [ ]:
from joblib import Parallel, delayed
from portfolio_test import portfolio_backtest_by_duration
from portfolio_test import make_benchmark_data

stats = pd.DataFrame(columns=['cagr', 'stdev', 'mdd', 'beta', 'alpha'])
example = Portfolio(stock_data, 10000, {'QQQ':50, 'SCHD':30, 'DGRW':20 })
benchmark = Portfolio(stock_data, 10000, {'QQQ':100})
duration = 5
cash_flow = 700

benchmark_data = make_benchmark_data(benchmark, example.start_date, duration, cash_flow)

param_list = []
for qqq_weight in range(0, 100, 5):
    for schd_weight in range(0, 10):
        ratio_qqq_etc = (qqq_weight, 100 - qqq_weight)
        ratio_schd_dgrw = (schd_weight, 10 - schd_weight)
        ratio_qqq_schd_dgrw = calc_target_ratio(ratio_qqq_etc, ratio_schd_dgrw)
        
        target_ratio = {
            'QQQ': ratio_qqq_schd_dgrw[0],
            'SCHD': ratio_qqq_schd_dgrw[1],
            'DGRW': ratio_qqq_schd_dgrw[2]
        }
        print(target_ratio)
        p = Portfolio(stock_data, 10000, target_ratio)
        param_list.append((p, benchmark_data, None, duration, cash_flow))
        break
    break

results = []
for params in param_list:
    results.append(portfolio_backtest_by_duration(*params))
# with Parallel(n_jobs=-1) as parallel:
#         results = parallel(delayed(portfolio_backtest_by_duration)(*params) for params in param_list)
        
for stat in results:
    stats = pd.concat([stats, stat])

stats

In [ ]:
stats.to_csv('./stats_qqq_schd_dgrw.csv')

In [ ]:
stats = pd.read_csv('./stats_qqq_schd_dgrw.csv', index_col=0)
stats

In [ ]:
weight_map = {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0,
              5: 1.1, 6: 1.1,
              7: 1.2, 8: 1.2}

stats['weight'] = stats.index.map(lambda x: weight_map[x % 9])
stats

def weighted_alpha_mean(group):
    d = group['alpha']
    w = group['weight']
    return (d * w).sum() / w.sum()

weighted_alpha_stat = stats.groupby('ratio').apply(weighted_alpha_mean)
weighted_alpha_stat = weighted_alpha_stat.sort_values(ascending=False)
mdd_stat = stats.groupby('ratio')['mdd'].quantile(0.25)

pd.set_option('display.max_rows', None)
mdd_stat

In [ ]:
stat_summary = stats.groupby(['ratio']).mean()
stat_summary['alpha'] = weighted_alpha_stat
stat_summary['mdd_median'] = mdd_stat
stat_summary['mdd_diff'] = stat_summary['mdd'] - stat_summary['mdd_median']
stat_summary = stat_summary.sort_values(by='alpha', ascending=False)
stat_summary = stat_summary[stat_summary['alpha'] > 0]
stat_summary.sort_values(by='mdd_diff', ascending=False, inplace=True)
stat_summary[(stat_summary['cagr'] > 14.5)]

In [ ]:
pd.set_option('display.max_rows', None)

stat_summary_sorted = stat_summary.sort_values(by=['alpha'], ascending=False)
stat_summary_sorted = stat_summary_sorted[stat_summary_sorted['alpha'] > 0]
stat_sorted = stats.sort_values(by=['ratio', 'alpha'], ascending=[True, False])

In [ ]:
import seaborn as sns

sns.catplot(data=stats, x='ratio', y='cagr', kind='box')
sns.catplot(data=stats, x='ratio', y='stdev', kind='box')
sns.catplot(data=stats, x='ratio', y='mdd', kind='box')

In [ ]:
sns.relplot(data=test, x='stdev', y='cagr', hue='ratio')

In [ ]:
sns.relplot(data=stats, x='stdev', y='cagr', hue='ratio')